In [16]:
import os
import sqlite3
import pandas as pd
from prefect import flow, task

# =====================================================================
# 1. CONFIGURACIÓN DEL ENTORNO LOCAL
# =====================================================================
# Desactivamos las llamadas de red de Prefect para que no busque servidores externos en Jupyter
os.environ["PREFECT_API_URL"] = ""
os.environ["PREFECT_API_DATABASE_CONNECTION_URL"] = "sqlite+aiosqlite:///:memory:"

DB_ORIGEN = "produccion_banco.db"
DB_DESTINO = "data_warehouse.db"

# =====================================================================
# 2. GENERACIÓN DE DATOS DE ORIGEN (SIMULACIÓN DE PRODUCCIÓN)
# =====================================================================
def inicializar_base_origen():
    conn = sqlite3.connect(DB_ORIGEN)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS transacciones_crudas (
            id_transaccion INTEGER, id_cliente TEXT, monto TEXT, tipo_operacion TEXT, fecha TEXT
        )
    """)
    datos = [
        (1001, "C-201", "$15,200.00", "transferencia", "2026-06-15"),
        (1002, "C-202", "$350.50", "tarjeta", "2026-06-16"),
        (1003, "C-203", None, "retiro", "2026-06-16"),         
        (1004, "C-204", "$85,000.00", "transferencia", "2026-06-17"), 
        (1005, "C-205", "$1,200.00", "tarjeta", "2026-06-17"),
        (1001, "C-201", "$15,200.00", "transferencia", "2026-06-15"), 
    ]
    cursor.execute("DELETE FROM transacciones_crudas")
    cursor.executemany("INSERT INTO transacciones_crudas VALUES (?, ?, ?, ?, ?)", datos)
    conn.commit()
    conn.close()

inicializar_base_origen()

# =====================================================================
# 3. COMPONENTES DEL PIPELINE ETL (PREFECT + PANDAS)
# =====================================================================
@task(retries=3, retry_delay_seconds=5)
def extract_from_source() -> pd.DataFrame:
    """E: Extracción de datos desde la base de datos de producción."""
    conn = sqlite3.connect(DB_ORIGEN)
    df = pd.read_sql_query("SELECT * FROM transacciones_crudas", conn)
    conn.close()
    return df

@task
def transform_and_evaluate_risk(df: pd.DataFrame, umbral_alerta: float = 40000.0) -> tuple[pd.DataFrame, pd.DataFrame]:
    """T: Limpieza profunda de datos, casteo de tipos y segmentación por riesgo."""
    # 1. Remover duplicados exactos
    df = df.drop_duplicates()
    
    # 2. Limpieza monetaria y tratamiento de valores nulos (Monto ausente = 0.0)
    df["monto"] = df["monto"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
    df["monto"] = pd.to_numeric(df["monto"], errors="coerce").fillna(0.0)
    
    # 3. Estandarización de formato de fechas
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.strftime("%Y-%m-%d")
    
    # 4. Regla de Negocio: Identificar transacciones sospechosas o que superan el umbral
    df["alto_riesgo"] = (df["monto"] >= umbral_alerta) | (df["monto"] == 0.0)
    
    df_normales = df[df["alto_riesgo"] == False].copy()
    df_alertas = df[df["alto_riesgo"] == True].copy()
    return df_normales, df_alertas

@task
def load_to_warehouse(df_normales: pd.DataFrame, df_alertas: pd.DataFrame):
    """L: Carga persistente en el Data Warehouse analítico."""
    conn = sqlite3.connect(DB_DESTINO)
    
    # Limpieza preventiva para evitar duplicar renglones si ejecutas la celda varias veces
    conn.cursor().execute("DROP TABLE IF EXISTS fact_transacciones_limpias")
    conn.cursor().execute("DROP TABLE IF EXISTS fact_alertas_riesgo")
    
    # Inserción en base de datos destino
    df_normales.to_sql("fact_transacciones_limpias", conn, if_exists="append", index=False)
    df_alertas.to_sql("fact_alertas_riesgo", conn, if_exists="append", index=False)
    conn.commit()
    conn.close()

@flow(name="Pipeline-ETL-Control-Riesgo")
def ejecutar_pipeline_financiero():
    """Orquestador principal del proceso ETL."""
    df_crudo = extract_from_source()
    df_clean, df_alerts = transform_and_evaluate_risk(df_crudo, umbral_alerta=40000.0)
    load_to_warehouse(df_clean, df_alerts)

# =====================================================================
# 4. EJECUCIÓN (BYPASS SEGURO PARA EL KERNEL DE JUPYTER)
# =====================================================================
print("⚙️ Ejecutando lógica ETL adaptada a Jupyter Notebook...")

# Invocamos la lógica interna (.fn) para evitar que el bucle asíncrono de Jupyter falle
df_crudo = extract_from_source.fn()
df_clean, df_alerts = transform_and_evaluate_risk.fn(df_crudo, umbral_alerta=40000.0)
load_to_warehouse.fn(df_clean, df_alerts)

print("🚀 ¡Lógica de datos y pipeline procesados con éxito de punta a punta!\n")

# =====================================================================
# 5. INSPECCIÓN COMPROBATORIA DE RESULTADOS EN JUPYTER
# =====================================================================
print("==================================================")
print("📊 RESULTADOS FINALES EN EL DATA WAREHOUSE")
print("==================================================")
conn_dw = sqlite3.connect(DB_DESTINO)

print("\n🔹 TABLA: fact_transacciones_limpias (Métricas comerciales limpias)")
display(pd.read_sql_query("SELECT * FROM fact_transacciones_limpias", conn_dw))

print("\n⚠️ TABLA: fact_alertas_riesgo (Aisladas para auditoría/cumplimiento)")
display(pd.read_sql_query("SELECT * FROM fact_alertas_riesgo", conn_dw))

conn_dw.close()

⚙️ Ejecutando lógica ETL adaptada a Jupyter Notebook...
🚀 ¡Lógica de datos y pipeline procesados con éxito de punta a punta!

📊 RESULTADOS FINALES EN EL DATA WAREHOUSE

🔹 TABLA: fact_transacciones_limpias (Métricas comerciales limpias)


,id_transaccion,id_cliente,monto,tipo_operacion,fecha,alto_riesgo
0,1001,C-201,15200.0,transferencia,2026-06-15,0
1,1002,C-202,350.5,tarjeta,2026-06-16,0
2,1005,C-205,1200.0,tarjeta,2026-06-17,0



⚠️ TABLA: fact_alertas_riesgo (Aisladas para auditoría/cumplimiento)


,id_transaccion,id_cliente,monto,tipo_operacion,fecha,alto_riesgo
0,1003,C-203,0.0,retiro,2026-06-16,1
1,1004,C-204,85000.0,transferencia,2026-06-17,1
